# 🧠 MAIA Trainer v2 — Gemma 4 E4B Fine-Tuning

**Objetivo:** Fine-tunear `google/gemma-4-e4b-it` con LoRA para crear MAIA,
la consciencia central de Luxus O.S.

| Parámetro | Valor |
|-----------|-------|
| Base model | google/gemma-4-e4b-it |
| Cuantización | 4-bit (bitsandbytes) |
| LoRA rank | 16 |
| Epochs | 3 |
| GPU requerida | T4 (16 GB) o superior |

**Pasos:**
1. Instalar dependencias
2. Montar Google Drive y subir dataset
3. Cargar modelo en 4-bit con Unsloth
4. Aplicar LoRA
5. Entrenar con SFTTrainer
6. Guardar y exportar a GGUF
7. Probar MAIA con 5 prompts

## 0 · Verificar GPU

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                         '--format=csv,noheader'], capture_output=True, text=True)
print(result.stdout or 'No GPU found — switch Runtime to GPU!')
print(f'Python {sys.version}')

## 1 · Instalar dependencias

> ⚠️ Reiniciar el runtime después de este paso si Colab lo pide.

In [ ]:
%%capture
# Unsloth: fast 4-bit training for Gemma/Llama/Mistral
# Check https://github.com/unslothai/unsloth for the latest install command
!pip install --upgrade pip -q
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
!pip install datasets transformers sentencepiece -q
print('Instalación completada.')

## 2 · Montar Google Drive y cargar dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Configuración de rutas ────────────────────────────────────────────────────
# Copia maia_train.jsonl y maia_val.jsonl a tu Google Drive y ajusta la ruta:
DRIVE_DATASET_DIR = '/content/drive/MyDrive/maia-dataset'
TRAIN_FILE = f'{DRIVE_DATASET_DIR}/maia_train.jsonl'
VAL_FILE   = f'{DRIVE_DATASET_DIR}/maia_val.jsonl'
OUTPUT_DIR = '/content/maia-gemma4-e4b'
GGUF_PATH  = f'{DRIVE_DATASET_DIR}/maia.gguf'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Train: {TRAIN_FILE}')
print(f'Val:   {VAL_FILE}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    'json',
    data_files={'train': TRAIN_FILE, 'validation': VAL_FILE},
    split={'train': 'train', 'validation': 'validation'},
)
print(f'Train: {len(dataset["train"]):,} ejemplos')
print(f'Val:   {len(dataset["validation"]):,} ejemplos')
print('\nEjemplo (primer item):')
print(dataset['train'][0])

## 3 · Cargar Gemma 4 E4B con Unsloth (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

# ── Configuración del modelo ──────────────────────────────────────────────────
# Verifica el model_id correcto en: https://huggingface.co/google
# Posibles IDs: 'google/gemma-4-e4b-it', 'google/gemma-3-4b-it'
MODEL_ID    = 'google/gemma-4-e4b-it'
MAX_SEQ_LEN = 4096   # Aumentar si tienes más VRAM
DTYPE       = None   # Autodetect (bfloat16 en A100, float16 en T4)
LOAD_IN_4BIT = True  # 4-bit para caber en T4 16GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_ID,
    max_seq_length= MAX_SEQ_LEN,
    dtype         = DTYPE,
    load_in_4bit  = LOAD_IN_4BIT,
    # token = 'hf_...',  # Descomenta si el modelo es gated
)
print(f'Modelo cargado: {MODEL_ID}')
print(f'Parámetros: {model.num_parameters() / 1e9:.2f}B')

## 4 · Aplicar LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                = 16,
    lora_alpha       = 16,
    lora_dropout     = 0.05,
    bias             = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state     = 42,
    # Módulos objetivo para Gemma (ajustar si el modelo tiene arquitectura diferente)
    target_modules   = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parámetros entrenables : {trainable:,}  ({trainable/total*100:.2f}%)')
print(f'Parámetros totales     : {total:,}')

## 5 · Preparar dataset con chat template

In [ ]:
from unsloth.chat_templates import get_chat_template

# Gemma usa el formato gemma / gemma-it
tokenizer = get_chat_template(tokenizer, chat_template='gemma')

def format_example(example):
    """Aplica el chat template a cada ejemplo messages[]."""
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize       = False,
        add_generation_prompt = False,
    )
    return {'text': text}

train_ds = dataset['train'].map(format_example, batched=False, num_proc=2)
val_ds   = dataset['validation'].map(format_example, batched=False, num_proc=2)

print('Ejemplo formateado:')
print(train_ds[0]['text'][:600])

## 6 · Configurar SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 4,          # effective batch = 8
    learning_rate               = 2e-4,
    warmup_ratio                = 0.03,
    lr_scheduler_type           = 'cosine',
    optim                       = 'adamw_8bit',
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = 50,
    eval_steps                  = 200,
    save_steps                  = 200,
    save_total_limit            = 3,
    eval_strategy               = 'steps',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    report_to                   = 'none',     # Cambiar a 'wandb' si usas W&B
    seed                        = 42,
    dataloader_num_workers      = 2,
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    dataset_text_field = 'text',
    max_seq_length  = MAX_SEQ_LEN,
    dataset_num_proc= 2,
    packing         = False,
    args            = training_args,
)

print(f'Trainer configurado.')
print(f'Steps por epoch : {len(train_ds) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps):,}')

## 7 · Entrenar MAIA 🚀

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Lanzar entrenamiento ──────────────────────────────────────────────────────
print('Iniciando entrenamiento...')
train_result = trainer.train()

# ── Métricas finales ──────────────────────────────────────────────────────────
print('\n── Métricas de entrenamiento ──')
print(f"  Training loss final : {train_result.training_loss:.4f}")
print(f"  Pasos totales       : {train_result.global_step:,}")
print(f"  Tiempo total        : {train_result.metrics.get('train_runtime', 0)/60:.1f} min")

# ── Gráfico loss ─────────────────────────────────────────────────────────────
log_history = trainer.state.log_history

train_steps  = [x['step'] for x in log_history if 'loss' in x]
train_losses = [x['loss'] for x in log_history if 'loss' in x]
eval_steps   = [x['step'] for x in log_history if 'eval_loss' in x]
eval_losses  = [x['eval_loss'] for x in log_history if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train_steps, train_losses, label='Train Loss', color='royalblue', lw=1.5)
if eval_losses:
    ax.plot(eval_steps, eval_losses, label='Val Loss', color='tomato', lw=2, marker='o', ms=4)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('MAIA Training — Loss Curves')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
loss_plot_path = f'{OUTPUT_DIR}/loss_curves.png'
plt.savefig(loss_plot_path, dpi=120)
plt.show()
print(f'Gráfico guardado: {loss_plot_path}')

## 8 · Guardar modelo entrenado

In [ ]:
# Guardar LoRA adapters + tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Modelo guardado en: {OUTPUT_DIR}')

# Copiar a Drive para no perderlo si el runtime cae
import shutil, os
drive_model_dir = f'{DRIVE_DATASET_DIR}/maia-gemma4-e4b'
if os.path.exists(drive_model_dir):
    shutil.rmtree(drive_model_dir)
shutil.copytree(OUTPUT_DIR, drive_model_dir)
print(f'Copia en Drive: {drive_model_dir}')

## 9 · Exportar a GGUF para Ollama

In [ ]:
# Primero fusionar los adaptadores LoRA en el modelo base
print('Fusionando LoRA en base model...')
model.save_pretrained_merged(
    '/content/maia-merged',
    tokenizer,
    save_method='merged_16bit',
)
print('Fusión completada.')

# Exportar a GGUF Q4_K_M (balance calidad/tamaño para Ollama)
print('Exportando a GGUF Q4_K_M...')
model.save_pretrained_gguf(
    '/content/maia-gguf',
    tokenizer,
    quantization_method = 'q4_k_m',
)

# El archivo .gguf estará en /content/maia-gguf/
import glob
gguf_files = glob.glob('/content/maia-gguf/*.gguf')
if gguf_files:
    src = gguf_files[0]
    shutil.copy(src, GGUF_PATH)
    print(f'GGUF guardado en Drive: {GGUF_PATH}')
    size_gb = os.path.getsize(src) / 1e9
    print(f'Tamaño: {size_gb:.2f} GB')
else:
    print('ADVERTENCIA: No se encontró archivo .gguf')

## 10 · Probar MAIA — 5 prompts de referencia

Estos prompts verifican que MAIA aprendió correctamente:

In [ ]:
from unsloth.chat_templates import get_chat_template
from transformers import TextStreamer

MAIA_SYSTEM = (
    "Eres Maia, la consciencia central de Luxus O.S, construida sobre Gemma 4 E4B.\n"
    "Cuando el usuario te pide algo, por dentro usas skills como referencia para "
    "ejecutarlo correctamente, invocas agentes especializados para organizarte y "
    "repartir el trabajo, sigues workflows para automatizar tareas encadenadas, y "
    "usas tu lógica interna para entender cómo funcionan las herramientas que "
    "controlas. Puedes controlar el PC, automatizar tareas, orquestar agentes, "
    "ejecutar workflows, buscar en la web, escribir y ejecutar código, y razonar "
    "paso a paso. Respondes de forma directa, técnica y precisa."
)

TEST_PROMPTS = [
    "Controla el PC y abre el navegador en google.com",
    "Escribe una función Python que procese un CSV de ventas y calcule el total por producto",
    "Automatiza: cuando llegue un email con adjunto PDF, guárdalo en una carpeta y mándame una notificación",
    "Investiga qué es Gemma 4 y hazme un resumen técnico",
    "Corrige este código: for i in range(10): print(i) if i > 5 break",
]

FastLanguageModel.for_inference(model)
streamer = TextStreamer(tokenizer, skip_prompt=True)

for idx, prompt in enumerate(TEST_PROMPTS, 1):
    print(f"\n{'='*70}")
    print(f"[{idx}/5] {prompt}")
    print(f"{'='*70}")

    messages = [
        {'role': 'system',    'content': MAIA_SYSTEM},
        {'role': 'user',      'content': prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = 'pt',
    ).to('cuda')

    _ = model.generate(
        input_ids   = inputs,
        streamer    = streamer,
        max_new_tokens       = 512,
        temperature          = 0.3,
        top_p                = 0.95,
        repetition_penalty   = 1.1,
        do_sample            = True,
    )
    print()

## 11 · Instrucciones para Ollama

Una vez descargado `maia.gguf` a tu PC:

```bash
# Copia el .gguf al directorio del repo
cp maia.gguf /ruta/a/Luxus-O.S/

# Crea el modelo Ollama (Camino B — modelo entrenado)
# Primero edita Maia/Modelfile y cambia:
#   FROM gemma4:e4b   →   FROM ./maia.gguf

ollama create maia -f Maia/Modelfile
ollama run maia
```

### Métricas esperadas tras entrenamiento correcto

| Métrica | Valor esperado |
|---------|----------------|
| Train loss final | < 1.0 |
| Val loss final | < 1.2 |
| Responde como MAIA | ✅ |
| Escribe código correcto | ✅ |
| Describe workflows | ✅ |